# AgroLab — análisis de resultados

Cuaderno del rol de **Analista de experimentación y datos**. Lee los CSV versionados en
`experiments/` y produce las tablas (`analysis/tablas/`) y figuras (`analysis/figuras/`)
del informe. Se ejecuta de punta a punta con:

```
jupyter nbconvert --to notebook --execute --inplace analysis/analisis.ipynb
```

Convenciones: cada configuración real tiene 5 repeticiones (semillas 1000–1004, las mismas en
todas las configuraciones: números aleatorios comunes). Se informa media, desvío estándar e
intervalo de confianza del 95 % con la distribución t de Student (Jain, 1991, cap. 13).

In [ ]:
import json, pathlib, tomllib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import stats

ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
EXP = ROOT / "experiments"
FIG = ROOT / "analysis" / "figuras"
TAB = ROOT / "analysis" / "tablas"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

params = tomllib.loads((EXP / "params.toml").read_text(encoding="utf-8"))
cal = json.loads((EXP / "calibration" / "calibration.json").read_text(encoding="utf-8"))

STRATS = ["rr", "ll", "ws"]
NAME = {"rr": "Round Robin", "ll": "Menos cargado", "ws": "Robo de trabajo"}
COLOR = {"rr": "#2a78d6", "ll": "#eb6834", "ws": "#1baf7a"}   # orden categórico fijo
HET = {"H0": "Homogéneo (60/60/60 %)", "H1": "Heterogéneo (100/50/30 %)"}
LATS = params["design"]["latencies_ms"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3de"

plt.rcParams.update({
    "font.family": ["Segoe UI", "DejaVu Sans"], "font.size": 10,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 220, "savefig.bbox": "tight",
})
coma = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", facecolor="white")

def ic95(x):
    x = np.asarray(x, dtype=float)
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def fmt(v, d=2):
    return f"{v:.{d}f}".replace(".", ",")

## 1. Calibración (paso 6)

In [ ]:
vel = pd.DataFrame(cal["worker_speed"]).T
vel.index.name = "nivel"
vel["coord"] = cal["coord_speed"]
print("Velocidad medida (unidades de trabajo por segundo; 1 u = SHA-256 de 32 KiB)")
display(vel.round(0))
ref = vel.loc["REF", ["w1", "w2", "w3"]]
rel = vel.loc[["H0", "H1"], ["w1", "w2", "w3"]].div(ref) * 100
print("Velocidad relativa a la misma VM sin límite (%)")
display(rel.round(1))
vel.round(1).to_csv(TAB / "calibracion_cpu.csv")

red = pd.DataFrame(cal["network"]).T
red.index.name = "netem_ms"
red["latency_ms"] = red["latency_s"] * 1000
red["bandwidth_MBps"] = red["bandwidth_Bps"] / 1e6
display(red[["latency_ms", "bandwidth_MBps", "r2"]].round(3))
red[["latency_ms", "bandwidth_MBps", "r2"]].round(4).to_csv(TAB / "calibracion_red.csv")
print(f"Costo del coordinador por mensaje: {cal['coord_msg_cost_s'] * 1e6:.0f} µs")
print(f"Decisión de Menos cargado: {cal['ll_decision']['base_s'] * 1e6:.2f} µs + "
      f"{cal['ll_decision']['per_worker_s'] * 1e9:.1f} ns por trabajador (r² = {cal['ll_decision']['r2']:.3f})")

In [ ]:
net = pd.read_csv(EXP / "calibration" / "network.csv")
base = net[net["size"] == 0].groupby(["netem_ms", "worker"])["rtt_s"].median() / 2
med = net.groupby(["netem_ms", "worker", "size"])["rtt_s"].median().reset_index()
med["ida_ms"] = (med["rtt_s"] - med.set_index(["netem_ms", "worker"]).index.map(base)) * 1000
fig, axes = plt.subplots(1, len(red), figsize=(10, 3), sharey=False)
for ax, (lat, g) in zip(axes, med.groupby("netem_ms")):
    fit = cal["network"][str(lat)]
    xs = np.linspace(0, g["size"].max(), 50)
    ax.plot(xs / 1024, (fit["latency_s"] + xs / fit["bandwidth_Bps"]) * 1000, color=INK2, lw=1.5, label="ajuste L + S/B")
    ax.scatter(g["size"] / 1024, g["ida_ms"], s=28, color="#2a78d6", edgecolor="white", linewidth=1, zorder=3, label="medido (mediana)")
    ax.set_title(f"netem {lat} ms")
    ax.set_xlabel("tamaño del mensaje (KiB)")
    ax.yaxis.set_major_formatter(coma)
axes[0].set_ylabel("tiempo de ida (ms)")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
guardar(fig, "fig_calibracion_red")

## 2. Resultados en el clúster real (paso 5)

In [ ]:
runs = pd.read_csv(EXP / "results" / "real" / "runs.csv")
METRICS = {
    "makespan_s": ("Makespan (s)", 2),
    "throughput_tps": ("Rendimiento (tareas/s)", 3),
    "mean_wait_s": ("Espera media (s)", 2),
    "cv_busy": ("Desbalance (CV de ocupación)", 3),
    "msgs_control": ("Mensajes de control", 0),
}
print(f"{len(runs)} corridas; código {runs['code'].unique()}; RTT medido por bloque (ms):")
display(runs.groupby(["latency_ms", "hetero"])["rtt_ms"].first().round(2).unstack())

g = runs.groupby(["hetero", "latency_ms", "strategy"])
resumen = g[list(METRICS)].agg(["mean", "std", ic95, "count"])
resumen.columns = [f"{m}_{s}" for m, s in resumen.columns]
resumen = resumen.reset_index()
resumen.to_csv(TAB / "real_resumen.csv", index=False)

tabla = resumen[["hetero", "latency_ms", "strategy"]].copy()
tabla["strategy"] = tabla["strategy"].map(NAME)
for m, (label, d) in METRICS.items():
    tabla[label] = [f"{fmt(a, d)} ± {fmt(b, d)}" for a, b in zip(resumen[f"{m}_mean"], resumen[f"{m}_std"])]
tabla.columns = ["Heterogeneidad", "Latencia (ms)", "Estrategia", *[v[0] for v in METRICS.values()]]
tabla.to_csv(TAB / "tabla_real.csv", index=False)
display(tabla)

In [ ]:
def barras(metric, ylabel, nombre, df=resumen, ic=True):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
    w = 0.26
    for ax, h in zip(axes, ["H0", "H1"]):
        d = df[df["hetero"] == h]
        for k, s in enumerate(STRATS):
            ds = d[d["strategy"] == s].set_index("latency_ms").reindex(LATS)
            x = np.arange(len(LATS)) + (k - 1) * (w + 0.02)
            err = ds[f"{metric}_ic95"] if ic else None
            ax.bar(x, ds[f"{metric}_mean"], width=w, color=COLOR[s], label=NAME[s], yerr=err,
                   error_kw={"ecolor": INK2, "elinewidth": 1, "capsize": 2})
        ax.set_xticks(np.arange(len(LATS)), [f"{l} ms" for l in LATS])
        ax.set_title(HET[h])
        ax.set_xlabel("latencia de red (netem)")
        ax.yaxis.set_major_formatter(coma)
    axes[0].set_ylabel(ylabel)
    axes[0].legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    guardar(fig, nombre)
    return fig

barras("makespan_s", "makespan (s), media ± IC 95 %", "fig_real_makespan");
barras("mean_wait_s", "espera media (s), media ± IC 95 %", "fig_real_espera");
barras("cv_busy", "CV de ocupación entre nodos", "fig_real_desbalance");

In [ ]:
# Prueba de diferencias entre estrategias en cada configuración (ANOVA de un factor
# sobre el makespan; con 5 repeticiones por grupo).
filas = []
for (h, lat), d in runs.groupby(["hetero", "latency_ms"]):
    grupos = [d.loc[d["strategy"] == s, "makespan_s"] for s in STRATS]
    f, p = stats.f_oneway(*grupos)
    medias = {s: x.mean() for s, x in zip(STRATS, grupos)}
    mejor = min(medias, key=medias.get)
    filas.append({"Heterogeneidad": h, "Latencia (ms)": lat, "F": round(f, 2), "p": p,
                  "Mejor estrategia": NAME[mejor]})
anova = pd.DataFrame(filas)
anova.to_csv(TAB / "real_anova.csv", index=False)
display(anova)

## 3. Validación del simulador

In [ ]:
val = pd.read_csv(EXP / "results" / "sim" / "validation.csv")
sim = val.groupby(["hetero", "latency_ms", "strategy"])[list(METRICS)].mean().reset_index()
comp = resumen.merge(sim, on=["hetero", "latency_ms", "strategy"], suffixes=("", "_sim"))
for m in ["makespan_s", "mean_wait_s", "throughput_tps", "cv_busy"]:
    comp[f"err_{m}"] = (comp[m] - comp[f"{m}_mean"]).abs() / comp[f"{m}_mean"]
cols = ["hetero", "latency_ms", "strategy", "makespan_s_mean", "makespan_s", "err_makespan_s",
        "mean_wait_s_mean", "mean_wait_s", "err_mean_wait_s", "throughput_tps_mean", "throughput_tps",
        "err_throughput_tps", "cv_busy_mean", "cv_busy", "err_cv_busy"]
comp[cols].to_csv(TAB / "validacion.csv", index=False)
v = comp[["hetero", "latency_ms", "strategy"]].copy()
v["strategy"] = v["strategy"].map(NAME)
v["Makespan real (s)"] = comp["makespan_s_mean"].map(fmt)
v["Makespan simulado (s)"] = comp["makespan_s"].map(fmt)
v["Error makespan (%)"] = (comp["err_makespan_s"] * 100).map(lambda x: fmt(x, 1))
v["Espera real (s)"] = comp["mean_wait_s_mean"].map(fmt)
v["Espera simulada (s)"] = comp["mean_wait_s"].map(fmt)
v["Error espera (%)"] = (comp["err_mean_wait_s"] * 100).map(lambda x: fmt(x, 1))
v.columns = ["Heterogeneidad", "Latencia (ms)", "Estrategia", *v.columns[3:]]
v.to_csv(TAB / "tabla_validacion.csv", index=False)
display(v)
print("Error relativo del makespan: media {:.1%}, mediana {:.1%}, máximo {:.1%}".format(
    comp["err_makespan_s"].mean(), comp["err_makespan_s"].median(), comp["err_makespan_s"].max()))
print("Error relativo de la espera media: media {:.1%}, máximo {:.1%}".format(
    comp["err_mean_wait_s"].mean(), comp["err_mean_wait_s"].max()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (m, label) in zip(axes, [("makespan_s", "makespan (s)"), ("mean_wait_s", "espera media (s)")]):
    lo = min(comp[f"{m}_mean"].min(), comp[m].min()) * 0.9
    hi = max(comp[f"{m}_mean"].max(), comp[m].max()) * 1.05
    ax.plot([lo, hi], [lo, hi], color=INK2, lw=1, ls="--", label="simulado = real")
    for s in STRATS:
        for h, marker in [("H0", "o"), ("H1", "s")]:
            d = comp[(comp["strategy"] == s) & (comp["hetero"] == h)]
            ax.errorbar(d[f"{m}_mean"], d[m], xerr=d[f"{m}_ic95"], fmt=marker, ms=7, color=COLOR[s],
                        mec="white", mew=1, ecolor=COLOR[s], elinewidth=1,
                        label=f"{NAME[s]} · {h}")
    ax.set_xlabel(f"{label} real (media ± IC 95 %)")
    ax.set_ylabel(f"{label} simulado")
    ax.xaxis.set_major_formatter(coma); ax.yaxis.set_major_formatter(coma)
axes[0].legend(fontsize=7.5, loc="upper left")
fig.tight_layout()
guardar(fig, "fig_validacion_dispersion")

fig, ax = plt.subplots(figsize=(10, 3.2))
w = 0.26
etiquetas = []
for i, (h, lat) in enumerate([(h, l) for h in ["H0", "H1"] for l in LATS]):
    etiquetas.append(f"{h}\n{lat} ms")
    for k, s in enumerate(STRATS):
        e = comp[(comp["hetero"] == h) & (comp["latency_ms"] == lat) & (comp["strategy"] == s)]["err_makespan_s"]
        ax.bar(i + (k - 1) * (w + 0.02), e.iloc[0] * 100, width=w, color=COLOR[s], label=NAME[s] if i == 0 else None)
ax.set_xticks(range(len(etiquetas)), etiquetas)
ax.set_ylabel("error relativo del makespan (%)")
ax.yaxis.set_major_formatter(coma)
ax.legend(fontsize=8, ncols=3, loc="upper left")
fig.tight_layout()
guardar(fig, "fig_validacion_error")

In [ ]:
# ¿El simulador conserva el orden de las estrategias? (lo que importa para decidir)
filas = []
for (h, lat), d in comp.groupby(["hetero", "latency_ms"]):
    real = list(d.sort_values("makespan_s_mean")["strategy"])
    simu = list(d.sort_values("makespan_s")["strategy"])
    tau = stats.kendalltau([real.index(s) for s in STRATS], [simu.index(s) for s in STRATS]).statistic
    filas.append({"Heterogeneidad": h, "Latencia (ms)": lat,
                  "Orden real": " < ".join(NAME[s] for s in real),
                  "Orden simulado": " < ".join(NAME[s] for s in simu),
                  "Misma mejor": real[0] == simu[0], "tau de Kendall": round(tau, 2)})
ranking = pd.DataFrame(filas)
ranking.to_csv(TAB / "validacion_ranking.csv", index=False)
display(ranking)

## 4. Extrapolación por simulación (paso 7)

In [ ]:
scale = pd.read_csv(EXP / "results" / "sim" / "scale.csv")
sc = scale.groupby(["n_workers", "hetero", "latency_ms", "strategy"]).agg(
    makespan=("makespan_s", "mean"), espera=("mean_wait_s", "mean"), cv=("cv_busy", "mean"),
    coord=("coord_util", "mean"), tareas=("tasks", "first"), sim_s=("sim_wall_s", "mean")).reset_index()
sc.to_csv(TAB / "escala_resumen.csv", index=False)
NS = sorted(sc["n_workers"].unique())

def curvas(col, ylabel, nombre, logy=False):
    fig, axes = plt.subplots(2, len(LATS), figsize=(11, 5.6), sharex=True, sharey="row" if not logy else False)
    for i, h in enumerate(["H0", "H1"]):
        for j, lat in enumerate(LATS):
            ax = axes[i, j]
            for s in STRATS:
                d = sc[(sc["hetero"] == h) & (sc["latency_ms"] == lat) & (sc["strategy"] == s)].sort_values("n_workers")
                ax.plot(d["n_workers"], d[col], color=COLOR[s], lw=2, marker="o", ms=5, mec="white", mew=1, label=NAME[s])
            ax.set_xscale("log", base=2)
            ax.set_xticks(NS, [str(n) for n in NS])
            if logy:
                ax.set_yscale("log")
            ax.yaxis.set_major_formatter(coma)
            ax.set_title(f"{h} · {lat} ms", fontsize=10)
            if i == 1:
                ax.set_xlabel("trabajadores simulados (N)")
        axes[i, 0].set_ylabel(ylabel)
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    guardar(fig, nombre)

curvas("makespan", "makespan (s)", "fig_escala_makespan")
curvas("cv", "CV de ocupación", "fig_escala_desbalance")
curvas("coord", "utilización del coordinador", "fig_escala_coordinador")

In [ ]:
# Ranking de estrategias en cada escala
filas = []
for (n, h, lat), d in sc.groupby(["n_workers", "hetero", "latency_ms"]):
    orden = list(d.sort_values("makespan")["strategy"])
    filas.append({"N": n, "Heterogeneidad": h, "Latencia (ms)": lat, "Mejor": NAME[orden[0]],
                  "Orden": " < ".join(NAME[s] for s in orden)})
rank_esc = pd.DataFrame(filas)
rank_esc.to_csv(TAB / "escala_ranking.csv", index=False)
display(rank_esc.pivot_table(index=["Heterogeneidad", "Latencia (ms)"], columns="N", values="Mejor", aggfunc="first"))

## 5. Consumo de recursos y costo de simular frente a emular

In [ ]:
real_ini = pd.to_datetime(runs["started"]).min()
real_fin = (pd.to_datetime(runs["started"]) + pd.to_timedelta(runs["wall_s"], unit="s")).max()
camp_real = (real_fin - real_ini).total_seconds()
costo = pd.DataFrame([
    {"Qué": "Campaña real (90 corridas, incluye cambios de condiciones)", "Tiempo de reloj (min)": camp_real / 60},
    {"Qué": "Suma de corridas reales", "Tiempo de reloj (min)": runs["wall_s"].sum() / 60},
    {"Qué": "Validación simulada (90 simulaciones)", "Tiempo de reloj (min)": val["sim_wall_s"].sum() / 60},
    {"Qué": "Extrapolación simulada (CPU acumulada)", "Tiempo de reloj (min)": scale["sim_wall_s"].sum() / 60},
])
costo["Tiempo de reloj (min)"] = costo["Tiempo de reloj (min)"].round(2)
costo.to_csv(TAB / "costo_simular_emular.csv", index=False)
display(costo)
print("Tiempo de simulación por corrida según N (s):")
display(sc.groupby("n_workers")["sim_s"].mean().round(2))

host_path = EXP / "results" / "real" / "host.csv"
if host_path.exists():
    host = pd.read_csv(host_path, parse_dates=["timestamp"])
    host = host[(host["timestamp"] >= real_ini) & (host["timestamp"] <= real_fin)]
    display(host[["host_cpu_pct", "host_free_mb", "vbox_cpu_pct", "vbox_mem_mb", "other_cpu_pct"]].describe().round(1))
    host.describe().round(2).to_csv(TAB / "anfitrion.csv")

vm_path = EXP / "results" / "real" / "vm_resources.csv"
if vm_path.exists():
    vm = pd.read_csv(vm_path)
    r = vm.groupby("node")[["cpu_cores", "mem_used_mb", "net_tx_kbps", "net_rx_kbps"]].agg(["mean", "max"]).round(2)
    r.to_csv(TAB / "vm_recursos.csv")
    display(r)